In [1]:
#Parameters
run_id = 5
aug_type = 0

In [2]:
model_name = "retfound_green"

In [3]:
# Multi image classification
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # allow multiple libiomp5md
os.environ["OMP_NUM_THREADS"] = "1"           # keep OpenMP under control

import torch
import albumentations as A
import numpy as np
import pandas as pd
#from .src.model import FoundationalCVModel, FoundationalCVModelWithClassifier
from model import UnifiedBackbone
from fundus_dataset import FundusDataset
from torch.utils.data import DataLoader

C:\ProgramData\Anaconda3\envs\mBRSETnew\lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [4]:
#data_dir = r'C:\\Users\\preet\\Documents\\mBRSET\\mBRSET_image_quality\\data\\'
dataset = "mBRSET" 

if dataset == "mBRSET":
    data_dir = r'C:\\Users\\preet\\Documents\\mBRSET\\mBRSET_image_quality\\data\\'
    train_df = pd.read_pickle(os.path.join(data_dir + 'mbrset_icdr_quality_2826_train_full.pkl'))
    val_df= pd.read_pickle(os.path.join(data_dir + 'mbrset_icdr_quality_2826_val_full.pkl'))
    test_df= pd.read_pickle(os.path.join(data_dir + 'mbrset_icdr_quality_2826_test_full.pkl'))
    # Remove any rows for which final_icdr is NaN
    train_df.dropna(subset=['final_icdr'], inplace=True)
    val_df.dropna(subset=['final_icdr'], inplace=True)
    test_df.dropna(subset=['final_icdr'], inplace=True)
    test_df.to_csv(data_dir + 'test_df_imgq_0_diagnosis.csv') 
    str_prefix="mBRSET_"


if dataset == "BRSET":
    data_dir = r'C:\\Users\\preet\\Documents\\BRSET\data\\'
    train_df = pd.read_csv(os.path.join(data_dir + 'train_df.csv'))
    val_df= pd.read_csv(os.path.join(data_dir + 'val_df.csv'))
    test_df= pd.read_csv(os.path.join(data_dir + 'test_df.csv'))
    # Remove any rows for which final_icdr is NaN
    train_df.dropna(subset=['final_icdr'], inplace=True)
    val_df.dropna(subset=['final_icdr'], inplace=True)
    test_df.dropna(subset=['final_icdr'], inplace=True)
    val_df = val_df[val_df["laterality"] == 0]
    test_df = test_df[test_df["laterality"] == 0]
    test_df.to_csv(data_dir + 'test_df_imgq_0_diagnosis.csv') 
    str_prefix="BRSET_"

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(2904, 6)
(668, 6)
(628, 6)


In [5]:
print(train_df["final_quality"].value_counts())

final_quality
1    2799
0     105
Name: count, dtype: int64


In [6]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

if model_name != "retfound_green":
    train_tf = A.Compose([
        A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(),
        A.VerticalFlip(),
        A.Normalize(mean=(0.485, 0.456, 0.406), 
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    val_tf = A.Compose([
        A.Resize(392, 392),
        A.Normalize(mean=(0.485, 0.456, 0.406), 
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
else:
    train_tf = A.Compose([
        A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(),
        A.VerticalFlip(),
        A.Normalize(mean=(0.5, 0.5, 0.5), 
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])
    
    val_tf = A.Compose([
        A.Resize(392, 392),
        A.Normalize(mean=(0.5, 0.5, 0.5), 
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

In [7]:
from diagnosis_train_eval import train_model

import numpy as np
import torch
import torch.nn as nn
def compute_class_weights(df, threshold=0):
    labels = (df["final_icdr"] > threshold).astype(int).values
    class_counts = np.bincount(labels)
    weights = 1.0 / class_counts
    weights = weights / weights.sum()
    return torch.tensor(weights, dtype=torch.float32)
    

device = "cuda"


In [ ]:
from model import UnifiedBackbone
from fundus_dataset import FundusDataset
from torch.utils.data import DataLoader

if dataset == "mBRSET":
    img_root = r"C:\\Users\\preet\\Documents\\mBRSET\\mbrset-a-mobile-brazilian-retinal-dataset-1.0\\images"

if dataset == "BRSET":
    img_root = r"C:\\Users\\preet\\Documents\\BRSET\\data\\resized_fundus_photos"

train_ds_single = FundusDataset(train_df,img_root, high_quality_tf=train_tf, low_quality_tf=train_tf, label_col='final_icdr')
val_ds_single = FundusDataset(val_df,img_root, high_quality_tf=val_tf, low_quality_tf=val_tf, label_col='final_icdr')
test_ds_single = FundusDataset(test_df,img_root, high_quality_tf=val_tf, low_quality_tf=val_tf, label_col='final_icdr')


device = "cuda"


train_loader = DataLoader(train_ds_single, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds_single,   batch_size=4, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds_single,  batch_size=4, shuffle=False, num_workers=0)


model = UnifiedBackbone(model_name = model_name)
class_weights = compute_class_weights(train_df).to(device)
print(class_weights)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)

In [ ]:


best_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=20,
    patience=5,
    str_prefix = str_prefix
)


## Tracing model for demo

In [ ]:
best_model = UnifiedBackbone(model_name = model_name)
best_model.load_state_dict(torch.load("mBRSET_img_diagnosis_model_392.pth"))
best_model.eval()

In [ ]:
import torch

x = torch.randn(1, 3, 392, 392)

In [ ]:
traced = torch.jit.trace(best_model, x)
traced.save(r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\results\mbrset_results" + r"\model.pt")
